# Description Topology Analysis

This notebook analyzes the relationship between semantic descriptions of channels and their point cloud (video embeddings) topology. It generates channel descriptions using Gemini, computes distances, and compares centrality.


## 1) Setup & Dependencies
Install requirements, mount Drive, and set up Google Colab environment variables.


In [ ]:
!pip install -q pandas numpy scipy pinecone-client google-generativeai sentence-transformers

import os
import json
import time
import pandas as pd
import numpy as np
import ast
from pathlib import Path
from scipy.spatial.distance import euclidean, cdist, pdist, squareform
from scipy.stats import pearsonr, spearmanr
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer

try:
    from google.colab import ai
except ImportError:
    pass

try:
    from google.colab import userdata
    from google.colab import drive
    drive.mount('/content/drive')
    is_colab = True
except ImportError:
    print('Not running in Colab. Falling back to local execution.')
    is_colab = False


## 2) Load 20D Embeddings Data
Load the 20D video embeddings from the Graphiko exports. Provide a fallback for local testing.


In [ ]:
DATA_PATH = Path('/content/drive/MyDrive/Graphiko/exports/video_embeddings_reduced/latest/business_cluster_video_embeddings_reduced_20d.csv')

if not DATA_PATH.exists():
    print(f"Warning: {DATA_PATH} not found. Creating a dummy dataset.")
    dummy_data = []
    for i in range(100):
        dummy_data.append({
            'video_id': f'vid_{i}',
            'channel_name': f'Channel_{i % 3}',
            'video_title': f'Video Title {i}',
            'embedding_20d': str([float(x) for x in np.random.randn(20)])
        })
    df = pd.DataFrame(dummy_data)
else:
    df = pd.read_csv(DATA_PATH)

if 'embedding_20d' in df.columns:
    if isinstance(df['embedding_20d'].iloc[0], str):
        df['embedding_20d'] = df['embedding_20d'].apply(ast.literal_eval)

print(f"Loaded {len(df)} videos across {df['channel_name'].nunique()} channels.")


## 3) Fetch or Generate Channel Descriptions
Group videos by channel and prompt Gemini to generate a description and segmented topics for each channel. Re-use existing results if found.


In [ ]:
output_dir = Path('/content/drive/MyDrive/research/')
if not is_colab:
    output_dir = Path('./research_output/')
output_dir.mkdir(parents=True, exist_ok=True)
descriptions_file = output_dir / 'channel_descriptions_and_segments.json'

def generate_description(channel_name, video_titles):
    if not is_colab:
        # Dummy response for local testing
        return {
            "description": f"Dummy description for {channel_name}.",
            "segments": [
                {"topic": "Core topic", "represented_videos": video_titles[:2]},
                {"topic": "Niche topic", "represented_videos": video_titles[2:4] if len(video_titles) > 2 else []}
            ]
        }
    prompt = f"""
    You are an expert content analyst. Analyze the following list of video titles for the YouTube channel '{channel_name}'.
    Titles: {', '.join(video_titles[:50])}  # Limit to avoid token limits

    Provide a JSON object with two keys:
    1. 'description': A comprehensive summary of the channel's main themes and style.
    2. 'segments': An ordered list of topics covered by the channel, starting with the most central/core ideas and moving towards more niche or outward areas. Each segment should include:
       - 'topic': A short name for the topic.
       - 'represented_videos': A list of up to 3 video titles from the provided list that best represent this topic.
    """
    try:
        time.sleep(2) # Respect rate limits
        # using gemini-1.5-pro
        response = ai.generate_text(prompt, model_name="google/gemini-1.5-pro")
        raw_text = response.text.strip()
        if raw_text.startswith('```json'):
            raw_text = raw_text[len('```json'):]
        if raw_text.endswith('```'):
            raw_text = raw_text[:-len('```')]
        return json.loads(raw_text)
    except Exception as e:
        print(f"Error generating description for {channel_name}: {e}")
        return None

channel_descriptions = {}
if descriptions_file.exists():
    with open(descriptions_file, 'r') as f:
        channel_descriptions = json.load(f)
    print("Loaded existing channel descriptions.")
else:
    print("Generating channel descriptions...")
    for channel_name, group in df.groupby('channel_name'):
        titles = group['video_title'].tolist()
        desc_data = generate_description(channel_name, titles)
        if desc_data:
            channel_descriptions[channel_name] = desc_data
    with open(descriptions_file, 'w') as f:
        json.dump(channel_descriptions, f, indent=2)
    print(f"Saved descriptions to {descriptions_file}")


## 4) Compare Relative Distances
Compute Point Cloud Distance (distance between channel centroids) and Description Distance (distance between text embeddings of descriptions).


In [ ]:
print("Loading SentenceTransformer model for text embeddings...")
st_model = SentenceTransformer('all-MiniLM-L6-v2')

channels = list(channel_descriptions.keys())

# Point Cloud Centroids
channel_centroids = {}
for ch in channels:
    ch_videos = df[df['channel_name'] == ch]['embedding_20d'].tolist()
    if ch_videos:
        channel_centroids[ch] = np.mean(ch_videos, axis=0)

# Description Embeddings
descriptions = [channel_descriptions[ch]['description'] for ch in channels]
desc_embeddings = st_model.encode(descriptions)
desc_emb_dict = {ch: emb for ch, emb in zip(channels, desc_embeddings)}

centroid_matrix = np.array([channel_centroids[ch] for ch in channels if ch in channel_centroids])
desc_matrix = np.array([desc_emb_dict[ch] for ch in channels if ch in desc_emb_dict])

if len(centroid_matrix) > 1:
    pc_distances = pdist(centroid_matrix, metric='euclidean')
    desc_distances = pdist(desc_matrix, metric='euclidean')
    
    pearson_corr, p_value = pearsonr(pc_distances, desc_distances)
    spearman_corr, s_p_value = spearmanr(pc_distances, desc_distances)
    
    print(f"Pearson correlation: {pearson_corr:.4f} (p={p_value:.4f})")
    print(f"Spearman correlation: {spearman_corr:.4f} (p={s_p_value:.4f})")
    
    plt.figure(figsize=(8, 6))
    plt.scatter(pc_distances, desc_distances, alpha=0.5)
    plt.xlabel('Point Cloud Distance (Centroids)')
    plt.ylabel('Description Embedding Distance')
    plt.title('Point Cloud Dist vs Description Dist')
    plt.grid(True)
    plt.show()


## 5) Compare Shape of Point Cloud with Description
Compare volume (dispersion) of point cloud vs volume of description segments, and check segment overlaps.


In [ ]:
pc_volumes = []
desc_volumes = []

all_segment_topics = []
all_segment_channel = []
all_segment_vids = []

for ch in channels:
    if ch not in channel_centroids: continue
    
    # Point Cloud Volume: average distance of videos to centroid
    ch_videos = np.array(df[df['channel_name'] == ch]['embedding_20d'].tolist())
    centroid = channel_centroids[ch]
    pc_vol = np.mean(np.linalg.norm(ch_videos - centroid, axis=1)) if len(ch_videos) > 0 else 0
    pc_volumes.append(pc_vol)
    
    # Description Volume: average distance of segment embeddings to overall description embedding
    segments = channel_descriptions[ch].get('segments', [])
    if segments:
        seg_topics = [s['topic'] for s in segments]
        seg_embs = st_model.encode(seg_topics)
        desc_emb = desc_emb_dict[ch]
        desc_vol = np.mean(np.linalg.norm(seg_embs - desc_emb, axis=1))
        
        for s in segments:
            all_segment_topics.append(s['topic'])
            all_segment_channel.append(ch)
            all_segment_vids.append(s.get('represented_videos', []))
    else:
        desc_vol = 0
    desc_volumes.append(desc_vol)

if len(pc_volumes) > 1:
    vol_corr, vol_p = pearsonr(pc_volumes, desc_volumes)
    print(f"Volume Correlation: {vol_corr:.4f} (p={vol_p:.4f})")
    
    plt.figure(figsize=(8, 6))
    plt.scatter(pc_volumes, desc_volumes, alpha=0.5, color='orange')
    plt.xlabel('Point Cloud Volume')
    plt.ylabel('Description Volume')
    plt.title('Point Cloud Volume vs Description Volume')
    plt.grid(True)
    plt.show()

# Segment Overlaps
if len(all_segment_topics) > 1:
    seg_embs = st_model.encode(all_segment_topics)
    
    # Compute pairwise distances for point clouds of these segments (using their represented videos)
    seg_pc_centroids = []
    valid_indices = []
    for i, vids in enumerate(all_segment_vids):
        vid_embs = df[df['video_title'].isin(vids)]['embedding_20d'].tolist()
        if vid_embs:
            seg_pc_centroids.append(np.mean(vid_embs, axis=0))
            valid_indices.append(i)
    
    if len(seg_pc_centroids) > 1:
        seg_pc_dists = pdist(np.array(seg_pc_centroids), metric='euclidean')
        valid_seg_embs = seg_embs[valid_indices]
        seg_desc_dists = pdist(valid_seg_embs, metric='euclidean')
        
        overlap_corr, overlap_p = spearmanr(seg_pc_dists, seg_desc_dists)
        print(f"Segment Overlap (PC Dist vs Desc Dist) Spearman: {overlap_corr:.4f} (p={overlap_p:.4f})")


## 6) Measure Centrality with Ordering
Calculate if earlier segments (central ideas) correspond to videos physically closer to the channel's centroid.


In [ ]:
segment_orders = []
segment_distances_to_centroid = []

for ch in channels:
    if ch not in channel_centroids: continue
    centroid = channel_centroids[ch]
    segments = channel_descriptions[ch].get('segments', [])
    
    for order, segment in enumerate(segments):
        rep_videos = segment.get('represented_videos', [])
        if not rep_videos: continue
        
        # Find these videos in the dataframe
        vid_embs = df[df['video_title'].isin(rep_videos) & (df['channel_name'] == ch)]['embedding_20d'].tolist()
        if vid_embs:
            avg_dist = np.mean(np.linalg.norm(vid_embs - centroid, axis=1))
            segment_orders.append(order)
            segment_distances_to_centroid.append(avg_dist)

if segment_orders:
    plt.figure(figsize=(8, 6))
    # Boxplot by order
    import seaborn as sns
    sns.boxplot(x=segment_orders, y=segment_distances_to_centroid)
    plt.xlabel('Segment Order (0 = First/Central)')
    plt.ylabel('Avg Distance of Represented Videos to Centroid')
    plt.title('Centrality vs Segment Order')
    plt.grid(True)
    plt.show()
    
    corr, p = spearmanr(segment_orders, segment_distances_to_centroid)
    print(f"Correlation between order and distance to centroid: {corr:.4f} (p={p:.4f})")
